# Recognising edited re-uploads at scale: reproduction notebook

This notebook is the code appendix of the report and reproduces every number in it from nothing but public data.
It is self-contained: it imports no code of ours from elsewhere, downloads the data itself, rebuilds the fixed splits and checks them against the ones the report used, and runs every experiment in order.

**What is ours and what is a library.**
The parts the report is about are implemented here: the edit grid that generates queries with known answers, bit-sampling LSH and MinHash LSH with banding, the s-curve predictions used to choose every LSH setting without searching, threshold calibration at a false-match rate, the jointly calibrated cascade, the cluster bootstrap, the failure analysis and the deduplication self-join.
Established building blocks come from libraries: the perceptual hash from `imagehash`, SIFT and RANSAC from OpenCV, k-means and the Davies-Bouldin index from scikit-learn, Louvain from networkx, and parallel feature extraction from joblib.
The DISC21 downloader is ours too: it reads single images out of Meta's 8.8 GB zip archives with parallel HTTP range requests instead of downloading all 350 GB.
The `remotezip` package offers the same single-member reads.

**How to run it.**

1. Install Python 3.12 and the libraries in the next cell (or `uv sync` in the project repository, which pins the same versions).
2. Set `DATASET` in the configuration cell: `"caltech101"` runs every experiment on Caltech-101 (a 130 MB download); `"disc21"` downloads about 21 GB of DISC21 and recreates every number in the report.
3. Run all cells. Features, vocabularies and thresholds are cached under `data/notebook-cache/`, so an interrupted run resumes where it stopped.

On a 32-thread machine with 64 GB of memory, the DISC21 experiments take about 10.5 hours after the download, most of it the test splits (208,000 queries each, every query searched by four systems exhaustively and through LSH); Caltech-101 takes an estimated 2 to 3 hours.
`QUERY_LIMIT = 100` turns either into a smoke test of a few minutes that exercises every cell.

**Licence.** DISC21 (Douze et al., 2021) may be used only for research on detecting manipulated media and may not be redistributed, so the images are downloaded from Meta's servers and never stored with this notebook; the failure gallery is written under `data/`, and the report shows at most 100 DISC21 images, each with its Flickr attribution.

In [ ]:
# pip install "numpy>=2.5" "pandas>=3.1" "scipy>=1.18" "scikit-learn>=1.9" "opencv-python-headless>=5.0" "pillow>=13.0" \
#     "imagehash>=4.3" "networkx>=3.7" "matplotlib>=3.11" "requests>=2.34" "tqdm>=4.70" "joblib>=1.6" "threadpoolctl>=3.7"

import csv
import hashlib
import io
import os
import struct
import tarfile
import threading
import time
import zipfile
import zlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path

import cv2
import imagehash
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
import scipy.sparse as sp
import scipy.special
from IPython.display import display
from joblib import Parallel, delayed, parallel_config
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageOps
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Configuration

The settings are the ones frozen on DISC21 validation (report, section 4): 8,192 visual words, MinHash with 256 permutations in 113 bands of 2 rows, bit sampling in 17 bands of 16 bits, and a RANSAC shortlist of the 10 best visual-word matches.
Section 9 shows the tuning that chose them.

In [ ]:
DATASET = "disc21"        # "caltech101" or "disc21"
QUERY_LIMIT = None        # None for the full splits; e.g. 100 for a smoke test (first N targets and N distractors)
WORKERS = max(1, (os.cpu_count() or 2) - 2)
DATA_DIR = Path(os.environ.get("NEAR_DUPLICATES_DATA", "data"))
CACHE = DATA_DIR / "notebook-cache" / DATASET
OUT = Path("results") / "notebook"
OUT.mkdir(parents=True, exist_ok=True)

SETTINGS = dict(vocabulary=8192, num_perm=256, minhash_bands=113, minhash_rows=2, bit_bands=17, bit_rows=16, top_k=10)
TARGET_FMR = 0.01
EXPERIMENTS = {
    "caltech101": dict(tune_vocabularies=[2048, 8192], scale_points=[2000, 4000], scale_split=False, dev=False,
                       dedup=dict(index_size=None, groups=300, copies=3)),
    "disc21": dict(tune_vocabularies=[2048, 8192], scale_points=[10_000, 25_000], scale_split=True, dev=True,
                   dedup=dict(index_size=10_000, groups=1000, copies=3)),
}[DATASET]

## 1. Data and the fixed splits

**Caltech-101** (Fei-Fei et al., 2004) is downloaded whole from CaltechDATA.
`Faces_easy` is excluded because it holds tight crops of the `Faces` images, which would be unlabelled near-duplicates.

**DISC21** holds 1,000,000 reference images and 50,000 development queries in zip archives of 8.8 GB each.
We read each archive's central directory once over HTTP and then fetch single images with one range request each, 32 at a time, so only the 122,500 images the splits name are transferred (about 21 GB with the scale split).

The splits are seeded, so they are rebuilt here rather than shipped, and their digests are checked against the splits the report used:

- `index`: the catalogue. For DISC21 the 10,000 references behind matched development queries plus 40,000 random others.
- `val_targets`, `test_targets`: 500 and 2,000 indexed images that become queries with a known answer.
- `val_distractors`, `test_distractors`: 500 and 2,000 images outside the index, edited the same way, to measure false matches.
- `dev_calibration`, `dev_report` (DISC21): Meta's development queries, half with a match and half without, split in two halves.
- `scale` (DISC21): 50,000 further references for the 100,000-image point.

Tune only on `val_*` and `dev_calibration`; report only `test_*` and `dev_report`.

In [ ]:
CALTECH_URL = "https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip?download=1"
DISC21_URL = "https://dl.fbaipublicfiles.com/image_similarity_challenge/public"
CALTECH_ROOT = DATA_DIR / "caltech101" / "101_ObjectCategories"
DISC21_DIR = DATA_DIR / "disc21"
DISC21_REFERENCES, DISC21_QUERIES, IMAGES_PER_ARCHIVE = 1_000_000, 50_000, 50_000
SPLIT_SEED = 2807
VAL_TARGETS, TEST_TARGETS, VAL_DISTRACTORS, TEST_DISTRACTORS = 500, 2000, 500, 2000
SPLIT_DIGESTS = {
    "caltech101": {"index": "a00057cadd0ce53b", "test_distractors": "2714770b2ef448bf", "test_targets": "a28be4f6e7a54683",
                   "val_distractors": "df0f3dd717e4966f", "val_targets": "29b5a34a0a0e1383"},
    "disc21": {"dev_calibration": "061bf93924fed89a", "dev_report": "73fcecc8032d6eeb", "index": "4376fe34518f3bbb",
               "scale": "7fa2b4919d5ff87c", "test_distractors": "97a26c9b41d8e85f", "test_targets": "dbb45c35ebff4f2f",
               "val_distractors": "710af7897684df0b", "val_targets": "fa279c7adb1a9a9c"},
}


def image_path(dataset: str, image_id: str) -> Path:
    if dataset == "caltech101":
        return CALTECH_ROOT / image_id
    return DISC21_DIR / ("references" if image_id.startswith("R") else "queries") / f"{image_id}.jpg"


def download_file(url: str, target: Path) -> Path:
    if target.exists():
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + ".part")
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(partial, "wb") as out:
            for chunk in tqdm(response.iter_content(1 << 20), desc=target.name, unit="MB"):
                out.write(chunk)
    os.replace(partial, target)
    return target


def fetch_range(session: requests.Session, url: str, first: int, last: int) -> bytes:
    response = session.get(url, headers={"Range": f"bytes={first}-{last}"}, timeout=60)
    response.raise_for_status()
    if response.status_code != 206:
        raise RuntimeError(f"{url} ignored the range request")
    return response.content


class HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of a remote file, so `zipfile` can read the archive's central directory."""

    def __init__(self, url: str, session: requests.Session):
        self.url, self.session, self.position = url, session, 0
        self.size = int(session.head(url, allow_redirects=True, timeout=60).headers["content-length"])

    def seekable(self) -> bool:
        return True

    def readable(self) -> bool:
        return True

    def seek(self, offset: int, whence: int = io.SEEK_SET) -> int:
        self.position = {io.SEEK_SET: 0, io.SEEK_CUR: self.position, io.SEEK_END: self.size}[whence] + offset
        return self.position

    def tell(self) -> int:
        return self.position

    def readinto(self, buffer) -> int:
        end = min(self.position + len(buffer), self.size)
        if end <= self.position:
            return 0
        data = fetch_range(self.session, self.url, self.position, end - 1)
        buffer[: len(data)] = data
        self.position += len(data)
        return len(data)


LOCAL_HEADER = struct.Struct("<4s5H3L2H")


class RemoteZip:
    """Single members of a remote zip archive, one range request each, safe to call from many threads."""

    def __init__(self, url: str):
        self.url, self.local = url, threading.local()
        with io.BufferedReader(HttpRangeFile(url, self.session()), buffer_size=1 << 20) as remote:
            self.members = {Path(info.filename).name: info for info in zipfile.ZipFile(remote).infolist()}

    def session(self) -> requests.Session:
        if not hasattr(self.local, "session"):
            self.local.session = requests.Session()
        return self.local.session

    def read(self, name: str) -> bytes:
        info = self.members[name]
        start = info.header_offset
        raw = fetch_range(self.session(), self.url, start, start + LOCAL_HEADER.size + 512 + info.compress_size - 1)
        signature, *_, name_length, extra_length = LOCAL_HEADER.unpack_from(raw)
        if signature != b"PK\x03\x04":
            raise RuntimeError(f"{name}: bad local header")
        data_start = LOCAL_HEADER.size + name_length + extra_length
        if data_start + info.compress_size > len(raw):
            raw = fetch_range(self.session(), self.url, start, start + data_start + info.compress_size - 1)
        data = raw[data_start : data_start + info.compress_size]
        if info.compress_type == zipfile.ZIP_DEFLATED:
            data = zlib.decompressobj(-zlib.MAX_WBITS).decompress(data)
        if zlib.crc32(data) != info.CRC:
            raise RuntimeError(f"{name}: checksum mismatch")
        return data


def fetch_members(archive_url: str, ids: list[str]) -> list[str]:
    """Download the given DISC21 images from one remote archive; returns those that failed after three attempts."""
    archive = RemoteZip(archive_url)

    def fetch(image_id: str) -> None:
        target = image_path("disc21", image_id)
        for attempt in range(3):
            try:
                data = archive.read(target.name)
                break
            except (requests.RequestException, RuntimeError):
                if attempt == 2:
                    raise
        partial = target.with_name(target.name + ".part")
        partial.write_bytes(data)
        os.replace(partial, target)

    failed = []
    with ThreadPoolExecutor(32) as pool:
        futures = {pool.submit(fetch, image_id): image_id for image_id in ids}
        for future in tqdm(as_completed(futures), total=len(futures), desc=Path(archive_url).name):
            if future.exception() is not None:
                failed.append(futures[future])
    return failed

In [ ]:
def checked(splits: dict[str, list[str]]) -> dict[str, list[str]]:
    index = set(splits["index"])
    assert all(set(splits[name]) <= index for name in ("val_targets", "test_targets"))
    assert not any(set(splits[name]) & index for name in ("val_distractors", "test_distractors"))
    for a, b in (("val_targets", "test_targets"), ("val_distractors", "test_distractors"), ("dev_calibration", "dev_report")):
        assert a not in splits or not set(splits[a]) & set(splits[b])
    return splits


def caltech_splits() -> dict[str, list[str]]:
    """Held-out images become distractor sources; validation and test targets are disjoint subsets of the index."""
    ids = sorted(p.relative_to(CALTECH_ROOT).as_posix() for p in CALTECH_ROOT.rglob("*.jpg") if p.parent.name != "Faces_easy")
    rng = np.random.default_rng(SPLIT_SEED)
    shuffled = [ids[i] for i in rng.permutation(len(ids))]
    held_out, index = shuffled[: VAL_DISTRACTORS + TEST_DISTRACTORS], shuffled[VAL_DISTRACTORS + TEST_DISTRACTORS :]
    targets = [index[i] for i in rng.permutation(len(index))[: VAL_TARGETS + TEST_TARGETS]]
    return checked({"index": sorted(index), "val_targets": sorted(targets[:VAL_TARGETS]), "test_targets": sorted(targets[VAL_TARGETS:]),
                    "val_distractors": sorted(held_out[:VAL_DISTRACTORS]), "test_distractors": sorted(held_out[VAL_DISTRACTORS:])})


def dev_ground_truth() -> dict[str, str]:
    """DISC21 development query -> the reference it was made from, for the queries that have one."""
    with open(DISC21_DIR / "dev_ground_truth.csv", newline="", encoding="utf-8-sig") as handle:
        rows = [tuple(value.strip() for value in row) for row in csv.reader(handle) if row]
    return {query: reference for query, reference in rows if reference and query != "query_id"}


def disc21_splits(truth: dict[str, str], index_size: int = 50_000, negative_queries: int = 10_000) -> dict[str, list[str]]:
    """Index every reference a matched development query was made from, plus random others up to `index_size`."""
    rng = np.random.default_rng(SPLIT_SEED)
    matched = sorted(set(truth.values()))
    others = np.setdiff1d(np.array([f"R{i:06d}" for i in range(DISC21_REFERENCES)]), matched)
    chosen = rng.permutation(others)
    extra = index_size - len(matched)
    index = matched + chosen[:extra].tolist()
    distractors = chosen[extra : extra + VAL_DISTRACTORS + TEST_DISTRACTORS].tolist()
    targets = [index[i] for i in rng.permutation(len(index))[: VAL_TARGETS + TEST_TARGETS]]
    positives = sorted(truth)
    negatives = rng.permutation(np.setdiff1d([f"Q{i:05d}" for i in range(DISC21_QUERIES)], positives))[:negative_queries].tolist()
    positives = rng.permutation(positives).tolist()
    splits = checked({"index": sorted(index), "val_targets": sorted(targets[:VAL_TARGETS]), "test_targets": sorted(targets[VAL_TARGETS:]),
                      "val_distractors": sorted(distractors[:VAL_DISTRACTORS]), "test_distractors": sorted(distractors[VAL_DISTRACTORS:]),
                      "dev_calibration": sorted(positives[::2] + negatives[::2]), "dev_report": sorted(positives[1::2] + negatives[1::2])})
    taken = sorted({i for name in ("index", "val_distractors", "test_distractors", "dev_calibration", "dev_report") for i in splits[name]})
    free = np.setdiff1d([f"R{i:06d}" for i in range(DISC21_REFERENCES)], taken)
    splits["scale"] = sorted(np.random.default_rng(SPLIT_SEED + 1).permutation(free)[:50_000].tolist())
    return splits


def digest(ids: list[str]) -> str:
    return hashlib.sha256(("\n".join(ids) + "\n").encode()).hexdigest()[:16]


def prepare(dataset: str) -> dict[str, list[str]]:
    """Download the dataset, rebuild its splits, check them against the report's, and fetch the images they name."""
    if dataset == "caltech101":
        if not CALTECH_ROOT.exists():
            archive = download_file(CALTECH_URL, CALTECH_ROOT.parent / "caltech-101.zip")
            with zipfile.ZipFile(archive) as outer:
                inner = next(name for name in outer.namelist() if name.endswith("101_ObjectCategories.tar.gz"))
                with outer.open(inner) as stream, tarfile.open(fileobj=stream, mode="r|gz") as tar:
                    tar.extractall(CALTECH_ROOT.parent, filter="data")
            archive.unlink()
        splits = caltech_splits()
    else:
        download_file(f"{DISC21_URL}/dev_ground_truth.csv", DISC21_DIR / "dev_ground_truth.csv")
        download_file(f"{DISC21_URL}/disc21_yfcc_attributions.csv", DISC21_DIR / "disc21_yfcc_attributions.csv")
        splits = disc21_splits(dev_ground_truth())
    mismatched = [name for name, ids in splits.items() if digest(ids) != SPLIT_DIGESTS[dataset][name]]
    assert not mismatched, f"splits differ from the report's: {mismatched}"
    if dataset == "disc21":
        wanted = sorted({i for ids in splits.values() for i in ids} - set(splits["val_targets"]) - set(splits["test_targets"]))
        missing = [i for i in wanted if not image_path(dataset, i).exists()]
        print(f"{len(wanted) - len(missing):,} of {len(wanted):,} DISC21 images present, fetching {len(missing):,}")
        for folder in ("references", "queries"):
            (DISC21_DIR / folder).mkdir(parents=True, exist_ok=True)
        by_archive = {}
        for image_id in missing:
            name = "dev_queries.zip" if image_id.startswith("Q") else f"references_{int(image_id[1:]) // IMAGES_PER_ARCHIVE}.zip"
            by_archive.setdefault(name, []).append(image_id)
        failed = [i for name, ids in sorted(by_archive.items()) for i in fetch_members(f"{DISC21_URL}/{name}", ids)]
        assert not failed, f"{len(failed)} images failed to download; rerun this cell to retry them"
    return splits


SPLITS = prepare(DATASET)
pd.Series({name: len(ids) for name, ids in SPLITS.items()}, name="images")

## 2. Queries with known answers: the edit grid

Every target and distractor is edited under 52 conditions: nine edit types at graded strengths, extended until each system fails, plus four chains that pair an edit defeating the global hash (crop, rotation, flip) with one defeating local features (compression, downscaling, blur), at three levels.
Edits are applied to the full-resolution image, as a re-upload edits the original.
Each query's random choices (crop position, caption colour) are seeded by the query alone, so every query is reproducible in any order and process.

In [ ]:
SINGLE_CONDITIONS = (
    ("clean", 0),
    *(("jpeg", quality) for quality in (90, 70, 50, 30, 10, 5)),        # JPEG quality
    *(("downscale", side) for side in (0.75, 0.5, 0.25, 0.15, 0.1)),    # fraction of each side kept
    *(("crop", area) for area in (0.9, 0.75, 0.5, 0.3, 0.15)),          # fraction of the area kept
    *(("rotate", degrees) for degrees in (2, 5, 10, 15, 30, 45)),       # degrees, canvas expanded
    *(("blur", sigma) for sigma in (1, 2, 4, 8)),                       # Gaussian sigma in pixels
    *(("brightness", gain) for gain in (0.25, 0.5, 1.0, 2.0)),          # brightness factor 1 + s
    *(("contrast", loss) for loss in (0.25, 0.5, 0.75, 0.9)),           # contrast factor 1 - s
    *(("overlay", area) for area in (0.05, 0.15, 0.3, 0.5)),            # fraction of the area covered by a caption band
    ("flip", 1),
)
CHAINS = {
    "crop+jpeg+overlay": ((("crop", 0.9), ("jpeg", 70), ("overlay", 0.05)), (("crop", 0.75), ("jpeg", 30), ("overlay", 0.15)),
                          (("crop", 0.5), ("jpeg", 10), ("overlay", 0.3))),
    "crop+downscale": ((("crop", 0.9), ("downscale", 0.75)), (("crop", 0.75), ("downscale", 0.5)), (("crop", 0.5), ("downscale", 0.25))),
    "rotate+blur": ((("rotate", 2), ("blur", 1)), (("rotate", 5), ("blur", 2)), (("rotate", 10), ("blur", 4))),
    "flip+jpeg": ((("flip", 1), ("jpeg", 70)), (("flip", 1), ("jpeg", 30)), (("flip", 1), ("jpeg", 10))),
}
CONDITIONS = SINGLE_CONDITIONS + tuple((chain, level) for chain, levels in CHAINS.items() for level in (1, 2, 3))
MIN_SIDE = 8


def query_seed(kind: str, strength: float, image_id: str) -> int:
    return zlib.crc32(f"{kind}|{strength}|{image_id}".encode())


def transform(img: Image.Image, kind: str, strength: float, rng: np.random.Generator) -> Image.Image:
    if kind in CHAINS:
        for step, step_strength in CHAINS[kind][int(strength) - 1]:
            img = transform(img, step, step_strength, rng)
        return img
    w, h = img.size
    if kind == "clean":
        return img
    if kind == "jpeg":
        buffer = io.BytesIO()
        img.save(buffer, "JPEG", quality=int(strength))
        return Image.open(buffer).convert("RGB")
    if kind == "downscale":
        return img.resize((max(MIN_SIDE, round(w * strength)), max(MIN_SIDE, round(h * strength))), Image.Resampling.BILINEAR)
    if kind == "crop":
        cw, ch = max(MIN_SIDE, round(w * np.sqrt(strength))), max(MIN_SIDE, round(h * np.sqrt(strength)))
        x, y = int(rng.integers(0, w - cw + 1)), int(rng.integers(0, h - ch + 1))
        return img.crop((x, y, x + cw, y + ch))
    if kind == "rotate":
        return img.rotate(strength, resample=Image.Resampling.BICUBIC, expand=True)
    if kind == "blur":
        return img.filter(ImageFilter.GaussianBlur(strength))
    if kind == "brightness":
        return ImageEnhance.Brightness(img).enhance(1 + strength)
    if kind == "contrast":
        return ImageEnhance.Contrast(img).enhance(1 - strength)
    if kind == "overlay":
        band_height = round(h * strength)
        band = Image.new("RGB", (w, band_height), tuple(int(c) for c in rng.integers(0, 256, 3)))
        ImageDraw.Draw(band).text((5, band_height // 3), "SHARED 1,000,000 TIMES", fill=(255, 255, 255))
        out = img.copy()
        out.paste(band, (0, int(rng.integers(0, h - band_height + 1))))
        return out
    if kind == "flip":
        return ImageOps.mirror(img)
    raise ValueError(kind)

## 3. The two representations

**pHash** (System A) reduces an image to 64 bits: grayscale at 32 x 32 pixels, a 2-D DCT, and the sign of the 8 x 8 lowest frequencies against their median (Zauner, 2010; `imagehash.phash`).
Low frequencies survive compression, scaling and blur, but not edits that move content.

**SIFT** (System B) keeps the 300 strongest keypoints of a copy of at most 512 pixels on its longer side (Lowe, 2004); descriptors are stored as bytes, which is lossless because OpenCV saturates them to 0-255.
Local patches survive crops and rotations, but blur and downscaling remove the keypoints themselves.

Features are extracted in parallel and cached per image list, so every later cell reuses them.

In [ ]:
SIFT_MAX_SIDE, SIFT_FEATURES = 512, 300


def decode(path: Path) -> Image.Image:
    """The full-resolution image with its EXIF orientation applied and transparency composited onto white."""
    with Image.open(path) as source:
        rgba = ImageOps.exif_transpose(source).convert("RGBA")
        return Image.alpha_composite(Image.new("RGBA", rgba.size, "white"), rgba).convert("RGB")


def image_features(img: Image.Image) -> tuple[np.uint64, np.ndarray, np.ndarray]:
    """pHash of the image as given; positions and descriptors of the strongest SIFT keypoints of its 512-pixel copy."""
    code = np.uint64(int(str(imagehash.phash(img)), 16))
    small = img.copy()
    small.thumbnail((SIFT_MAX_SIDE, SIFT_MAX_SIDE), Image.Resampling.LANCZOS)
    sift = cv2.SIFT_create(nfeatures=SIFT_FEATURES, contrastThreshold=0.04, edgeThreshold=10.0, sigma=1.6)
    keypoints, descriptors = sift.detectAndCompute(cv2.cvtColor(np.asarray(small), cv2.COLOR_RGB2GRAY), None)
    order = sorted(range(len(keypoints)), key=lambda i: (-keypoints[i].response, *keypoints[i].pt, keypoints[i].size,
                                                          keypoints[i].angle, keypoints[i].octave))[:SIFT_FEATURES]
    if not order:
        return code, np.zeros((0, 2), np.float32), np.zeros((0, 128), np.uint8)
    return (code, np.array([keypoints[i].pt for i in order], np.float32).reshape(-1, 2),
            np.asarray(descriptors[order], np.float32).astype(np.uint8))


def extract_job(dataset: str, image_id: str, kind: str, strength: float):
    cv2.setNumThreads(1)
    rng = np.random.default_rng(query_seed(kind, strength, image_id))
    return image_features(transform(decode(image_path(dataset, image_id)), kind, strength, rng))


@dataclass
class FeatureSet:
    ids: list[str]
    hashes: np.ndarray
    descriptors: list[np.ndarray]
    keypoints: list[np.ndarray]

    def subset(self, positions: np.ndarray) -> "FeatureSet":
        return FeatureSet([self.ids[i] for i in positions], self.hashes[positions],
                          [self.descriptors[i] for i in positions], [self.keypoints[i] for i in positions])

    def concatenate(self, other: "FeatureSet") -> "FeatureSet":
        return FeatureSet(self.ids + other.ids, np.concatenate([self.hashes, other.hashes]),
                          self.descriptors + other.descriptors, self.keypoints + other.keypoints)


def cached_features(name: str, jobs: list[tuple], label: str) -> FeatureSet:
    """Features of `jobs` (dataset, image ID, edit, strength), extracted in parallel once and then read from the cache."""
    path = CACHE / f"{name}.npz"
    ids = [job[1] for job in jobs]
    if path.exists():
        with np.load(path) as cached:
            if cached["ids"].tolist() == ids:
                cut = cached["offsets"][1:-1]
                return FeatureSet(ids, cached["hashes"], np.split(cached["descriptors"], cut), np.split(cached["keypoints"], cut))
    # One BLAS thread per worker: each would otherwise reserve buffers for every core when it imports NumPy.
    with parallel_config(backend="loky", inner_max_num_threads=1):
        results = list(tqdm(Parallel(n_jobs=WORKERS, batch_size=16, return_as="generator")(delayed(extract_job)(*job) for job in jobs),
                            total=len(jobs), desc=label))
    features = FeatureSet(ids, np.array([r[0] for r in results], np.uint64), [r[2] for r in results], [r[1] for r in results])
    path.parent.mkdir(parents=True, exist_ok=True)
    offsets = np.concatenate([[0], np.cumsum([len(d) for d in features.descriptors])])
    partial = path.with_name(path.stem + ".part.npz")
    np.savez(partial, ids=np.array(ids), hashes=features.hashes, offsets=offsets,
             descriptors=np.concatenate(features.descriptors) if ids else np.zeros((0, 128), np.uint8),
             keypoints=np.concatenate(features.keypoints) if ids else np.zeros((0, 2), np.float32))
    os.replace(partial, path)
    return features


def index_features(scale: bool = False) -> FeatureSet:
    full = cached_features(f"index_n{len(SPLITS['index'])}", [(DATASET, i, "clean", 0) for i in SPLITS["index"]], "index")
    if scale:
        full = full.concatenate(cached_features(f"scale_n{len(SPLITS['scale'])}", [(DATASET, i, "clean", 0) for i in SPLITS["scale"]], "scale"))
    return full


INDEX = index_features()
example = decode(image_path(DATASET, SPLITS["val_targets"][0]))
rows = []
for kind, strength in [("clean", 0), ("jpeg", 10), ("downscale", 0.25), ("blur", 4), ("crop", 0.5), ("rotate", 10), ("flip", 1)]:
    code, points, _ = image_features(transform(example, kind, strength, np.random.default_rng(0)))
    rows.append(dict(edit=f"{kind} {strength}", hamming_to_source=int(np.bitwise_count(code ^ INDEX.hashes[SPLITS["index"].index(SPLITS["val_targets"][0])])),
                     keypoints=len(points)))
pd.DataFrame(rows).set_index("edit")  # unrelated images are about 32 bits apart

## 4. Visual words: the k-means vocabulary

Each SIFT descriptor is mapped to its nearest of k centroids, and an image becomes the set of its visual words (Sivic and Zisserman, 2003).
The vocabulary is fitted with mini-batch k-means (Sculley, 2010) on 300,000 descriptors sampled from the fitting pool: the indexed images that are no validation, test or development source, so no evaluation image shapes the words.
Its compactness is measured by the Davies-Bouldin index on a sample of 200,000 descriptors (lower is better; Davies and Bouldin, 1979).
Assignment is exact: the nearest centre by Euclidean distance, in batches.

In [ ]:
def fit_pool_ids() -> list[str]:
    excluded = set(SPLITS["val_targets"]) | set(SPLITS["test_targets"])
    if DATASET == "disc21":
        excluded |= set(dev_ground_truth().values())
    return sorted(set(SPLITS["index"]) - excluded)


@dataclass
class Vocabulary:
    centers: np.ndarray
    davies_bouldin: float

    def assign(self, descriptors: np.ndarray, batch: int = 8192) -> np.ndarray:
        norms = np.einsum("ij,ij->i", self.centers, self.centers)
        words = np.empty(len(descriptors), np.int32)
        for start in range(0, len(descriptors), batch):
            block = np.asarray(descriptors[start : start + batch], np.float32)
            words[start : start + len(block)] = np.argmin(norms[None, :] - 2 * block @ self.centers.T, axis=1)
        return words


def vocabulary(k: int, seed: int) -> Vocabulary:
    path = CACHE / f"vocabulary_k{k}_s{seed}.npz"
    if path.exists():
        with np.load(path) as cached:
            return Vocabulary(cached["centers"], float(cached["davies_bouldin"]))
    position = {image_id: i for i, image_id in enumerate(INDEX.ids)}
    pool = np.concatenate([INDEX.descriptors[position[i]] for i in fit_pool_ids()])
    sample = pool[np.sort(np.random.default_rng(seed).choice(len(pool), size=min(len(pool), 300_000), replace=False))].astype(np.float32)
    model = MiniBatchKMeans(n_clusters=k, batch_size=max(2048, k), random_state=seed, max_iter=100, n_init=3,
                            max_no_improvement=10, reassignment_ratio=0.01)
    rows = np.random.default_rng(seed).choice(len(sample), min(200_000, len(sample)), replace=False)
    with threadpool_limits(limits=1):
        model.fit(sample)
        labels = model.predict(sample[rows])
        db = davies_bouldin_score(sample[rows], labels)
    np.savez(path, centers=model.cluster_centers_.astype(np.float32), davies_bouldin=db)
    return Vocabulary(model.cluster_centers_.astype(np.float32), float(db))


def word_sets(descriptor_lists: list[np.ndarray], vocab: Vocabulary) -> list[np.ndarray]:
    sizes = [len(d) for d in descriptor_lists]
    words = vocab.assign(np.concatenate(descriptor_lists)) if sum(sizes) else np.empty(0, np.int32)
    return [np.unique(w) for w in np.split(words, np.cumsum(sizes)[:-1])]


def incidence(sets: list[np.ndarray], k: int) -> sp.csr_matrix:
    """Binary image-by-word matrix, so intersections with the whole index are one sparse product."""
    rows = np.repeat(np.arange(len(sets)), [len(s) for s in sets])
    columns = np.concatenate(sets) if sets else np.empty(0, np.int64)
    return sp.csr_matrix((np.ones(len(rows), np.float32), (rows, columns)), shape=(len(sets), k))

## 5. Locality-sensitive hashing (implemented here)

Both LSH families split a signature into b bands of r rows; two images become candidates if all rows of at least one band agree (Leskovec et al., 2020, chapter 3).

- **Bit sampling** for pHash (Indyk and Motwani, 1998): each band reads r distinct random bit positions. Two codes d bits apart agree on a band with probability C(64 - d, r) / C(64, r), slightly below the (1 - d/64)^r of the textbook s-curve, because positions are drawn without replacement.
- **MinHash** for visual-word sets (Broder, 1997): each of 256 exact random permutations of the vocabulary maps a set to its smallest rank, and two sets agree on a row with probability equal to their Jaccard similarity.

A pair with per-row agreement s is a candidate with probability 1 - (1 - s^r)^b.
All bands of all items live in one sorted table, so a query is one binary search per band.

In [ ]:
def s_curve(similarity, bands: int, rows: int) -> np.ndarray:
    return 1.0 - (1.0 - np.asarray(similarity, float) ** rows) ** bands


def bit_sampling_curve(distance, bands: int, rows: int) -> np.ndarray:
    per_band = scipy.special.comb(64 - np.asarray(distance), rows) / scipy.special.comb(64, rows)
    return 1.0 - (1.0 - per_band) ** bands


class BitSampler:
    def __init__(self, bands: int, rows: int, seed: int):
        rng = np.random.default_rng(seed)
        self.positions = np.stack([rng.choice(64, size=rows, replace=False) for _ in range(bands)])
        self.rows = rows

    def keys(self, codes: np.ndarray) -> np.ndarray:
        bits = (np.asarray(codes, np.uint64)[:, None, None] >> self.positions.astype(np.uint64)[None]) & np.uint64(1)
        return (bits * (np.uint64(1) << np.arange(self.rows, dtype=np.uint64))).sum(axis=2, dtype=np.uint64)


class MinHasher:
    EMPTY = np.iinfo(np.uint32).max

    def __init__(self, vocabulary_size: int, num_perm: int, seed: int):
        rng = np.random.default_rng(seed)
        self.ranks = np.array([rng.permutation(vocabulary_size) for _ in range(num_perm)], dtype=np.uint32)

    def transform(self, sets: list[np.ndarray]) -> np.ndarray:
        return np.array([self.ranks[:, s].min(axis=1) if len(s) else np.full(len(self.ranks), self.EMPTY, np.uint32) for s in sets],
                        dtype=np.uint32).reshape(-1, len(self.ranks))


def band_keys(signatures: np.ndarray, bands: int, rows: int, seed: int) -> np.ndarray:
    """One 64-bit key per band; the odd mixers are invertible modulo 2^64, so any changed value changes the key."""
    mixers = np.random.default_rng(seed).integers(1, 2**63, size=rows, dtype=np.uint64) | np.uint64(1)
    blocks = signatures[:, : bands * rows].astype(np.uint64).reshape(len(signatures), bands, rows)
    with np.errstate(over="ignore"):
        return (blocks * mixers).sum(axis=2, dtype=np.uint64)


def band_tagged(keys: np.ndarray) -> np.ndarray:
    with np.errstate(over="ignore"):
        return keys.astype(np.uint64) * np.uint64(0x9E3779B97F4A7C15) + np.arange(keys.shape[-1], dtype=np.uint64)


class BandIndex:
    """All (band, key) pairs in one sorted table; a query returns every item sharing at least one band key."""

    def __init__(self, keys: np.ndarray):
        n, bands = keys.shape
        tagged = band_tagged(keys).ravel()
        order = np.argsort(tagged, kind="stable")
        self.sorted_keys, self.items = tagged[order], (order // bands).astype(np.int32 if n < 2**31 else np.int64)

    def candidates(self, query_keys: np.ndarray) -> np.ndarray:
        tagged = band_tagged(np.asarray(query_keys)[None])[0]
        starts = np.searchsorted(self.sorted_keys, tagged, side="left")
        counts = np.searchsorted(self.sorted_keys, tagged, side="right") - starts
        if counts.sum() == 0:
            return np.empty(0, np.int64)
        offsets = np.cumsum(counts) - counts
        return np.unique(self.items[np.arange(counts.sum()) + np.repeat(starts - offsets, counts)]).astype(np.int64)

    @property
    def nbytes(self) -> int:
        return self.sorted_keys.nbytes + self.items.nbytes


fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.5))
d = np.arange(65)
left.plot(d, bit_sampling_curve(d, SETTINGS["bit_bands"], SETTINGS["bit_rows"]))
left.set(xlabel="Hamming distance", ylabel="P(candidate)", title=f"bit sampling, b={SETTINGS['bit_bands']}, r={SETTINGS['bit_rows']}")
s = np.linspace(0, 1, 201)
right.plot(s, s_curve(s, SETTINGS["minhash_bands"], SETTINGS["minhash_rows"]))
right.set(xlabel="Jaccard similarity", title=f"MinHash, b={SETTINGS['minhash_bands']}, r={SETTINGS['minhash_rows']}")
plt.show()

## 6. The four systems and spatial verification

Each system answers a query with its best-scoring indexed image, searched exhaustively and through its LSH index, so the loss LSH causes is measured directly.
Scores are "higher is more similar" for all of them, so one threshold rule serves every system.

| System | Score | LSH |
| --- | --- | --- |
| `phash` | minus the Hamming distance | bit sampling |
| `visual-words` | Jaccard similarity of the word sets | MinHash |
| `visual-words+ransac` | inliers of a RANSAC homography, re-ranking the 10 best by Jaccard | MinHash |
| `visual-words-containment` | share of the query's words found in the image | MinHash |

Spatial verification (Philbin et al., 2007) matches descriptors with Lowe's ratio test (0.75), lets each reference keypoint vote once, and fits a homography with RANSAC (Fischler and Bolles, 1981; OpenCV, 3-pixel reprojection error); a pair is accepted with at least 6 inliers making up at least a quarter of the matches, and scores its inlier count.

In [ ]:
def verify(query_points, reference_points, query_descriptors, reference_descriptors) -> tuple[int, int]:
    """(unique ratio-test matches, inliers of an accepted homography or 0) between two images."""
    if len(query_descriptors) < 4 or len(reference_descriptors) < 4:
        return 0, 0
    pairs = cv2.BFMatcher(cv2.NORM_L2).knnMatch(np.asarray(query_descriptors, np.float32), np.asarray(reference_descriptors, np.float32), k=2)
    good = [p[0] for p in pairs if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    unique, query_seen, reference_seen = {}, set(), set()
    for m in sorted(good, key=lambda m: (m.distance, m.queryIdx, m.trainIdx)):
        q, r = tuple(query_points[m.queryIdx]), tuple(reference_points[m.trainIdx])
        if m.trainIdx in unique or q in query_seen or r in reference_seen:
            continue
        unique[m.trainIdx] = m
        query_seen.add(q)
        reference_seen.add(r)
    good = list(unique.values())
    if len(good) < 4:
        return len(good), 0
    src = np.array([query_points[m.queryIdx] for m in good], np.float32)
    dst = np.array([reference_points[m.trainIdx] for m in good], np.float32)
    if np.linalg.matrix_rank(src - src.mean(axis=0)) < 2 or np.linalg.matrix_rank(dst - dst.mean(axis=0)) < 2:
        return len(good), 0
    cv2.setRNGSeed(2807)
    matrix, mask = cv2.findHomography(src, dst, cv2.RANSAC, 3.0, maxIters=2000, confidence=0.995)
    if matrix is None or mask is None or not np.isfinite(matrix).all() or np.linalg.matrix_rank(matrix) < 3:
        return len(good), 0
    inliers = int(mask.sum())
    return len(good), inliers if inliers >= 6 and inliers / len(good) >= 0.25 else 0


def top(scores: np.ndarray, positions: np.ndarray | None) -> tuple[int, float]:
    best = int(np.argmax(scores))
    return (best if positions is None else int(positions[best])), float(scores[best])


class PhashSystem:
    name = "phash"

    def __init__(self, hashes: np.ndarray, bands: int, rows: int, seed: int):
        self.hashes, self.sampler = hashes, BitSampler(bands, rows, seed)
        self.index = BandIndex(self.sampler.keys(hashes))

    def encode(self, features: FeatureSet):
        return features.hashes

    def keys(self, hashes):
        return self.sampler.keys(hashes)

    def scores(self, hashes, query: int, positions=None) -> np.ndarray:
        codes = self.hashes if positions is None else self.hashes[positions]
        return -np.bitwise_count(codes ^ hashes[query]).astype(float)

    def rank(self, hashes, query: int, positions=None):
        return top(self.scores(hashes, query, positions), positions)

    def similarity(self, hashes, query: int, position: int) -> float:
        return 1 + float(self.scores(hashes, query, np.array([position]))[0]) / 64

    def memory_bytes(self) -> int:
        return self.hashes.nbytes + self.index.nbytes


class VisualWordSystem:
    name = "visual-words"

    def __init__(self, descriptors: list[np.ndarray], vocab: Vocabulary, num_perm: int, bands: int, rows: int, seed: int):
        self.vocab, self.bands, self.rows, self.seed, self.words = vocab, bands, rows, seed, len(vocab.centers)
        sets = word_sets(descriptors, vocab)
        self.matrix = incidence(sets, self.words)
        self.sizes = np.asarray(self.matrix.sum(axis=1)).ravel()
        self.minhasher = MinHasher(self.words, num_perm, seed)
        self.index = BandIndex(self.keys(sets))

    def encode(self, features: FeatureSet):
        return word_sets(features.descriptors, self.vocab)

    def keys(self, sets):
        return band_keys(self.minhasher.transform(sets), self.bands, self.rows, seed=self.seed + 1)

    def intersections(self, sets, query: int, positions):
        matrix, sizes = (self.matrix, self.sizes) if positions is None else (self.matrix[positions], self.sizes[positions])
        indicator = np.zeros(self.words, np.float32)
        indicator[sets[query]] = 1
        return matrix @ indicator, sizes

    def scores(self, sets, query: int, positions=None) -> np.ndarray:
        intersection, sizes = self.intersections(sets, query, positions)
        return intersection / np.maximum(sizes + len(sets[query]) - intersection, 1)

    def rank(self, sets, query: int, positions=None):
        return top(self.scores(sets, query, positions), positions)

    def similarity(self, sets, query: int, position: int) -> float:
        return float(self.scores(sets, query, np.array([position]))[0])

    def memory_bytes(self) -> int:
        matrix = self.matrix.data.nbytes + self.matrix.indices.nbytes + self.matrix.indptr.nbytes
        return matrix + self.sizes.nbytes + self.vocab.centers.nbytes + self.minhasher.ranks.nbytes + self.index.nbytes


class ContainmentSystem:
    name = "visual-words-containment"

    def __init__(self, base: VisualWordSystem):
        self.base, self.index = base, base.index

    def encode(self, features):
        return self.base.encode(features)

    def keys(self, sets):
        return self.base.keys(sets)

    def scores(self, sets, query: int, positions=None) -> np.ndarray:
        return self.base.intersections(sets, query, positions)[0] / max(len(sets[query]), 1)

    def rank(self, sets, query: int, positions=None):
        return top(self.scores(sets, query, positions), positions)

    def similarity(self, sets, query: int, position: int) -> float:
        return float(self.scores(sets, query, np.array([position]))[0])

    def memory_bytes(self) -> int:
        return self.base.memory_bytes()


class SpatiallyVerifiedSystem:
    """B's top `top_k` by Jaccard re-ranked by inliers; the Jaccard similarity (below 1) only breaks ties."""

    name = "visual-words+ransac"

    def __init__(self, base: VisualWordSystem, descriptors, keypoints, top_k: int):
        self.base, self.index, self.top_k, self.descriptors, self.keypoints = base, base.index, top_k, descriptors, keypoints

    def encode(self, features):
        return self.base.encode(features), features

    def keys(self, encoded):
        return self.base.keys(encoded[0])

    def inliers(self, features: FeatureSet, query: int, position: int) -> int:
        return verify(features.keypoints[query], self.keypoints[position], features.descriptors[query], self.descriptors[position])[1]

    def rank(self, encoded, query: int, positions=None):
        sets, features = encoded
        jaccard = self.base.scores(sets, query, positions)
        shortlist = np.argsort(-jaccard, kind="stable")[: self.top_k]
        shortlist_positions = shortlist if positions is None else positions[shortlist]
        scores = np.array([self.inliers(features, query, int(p)) for p in shortlist_positions]) + jaccard[shortlist]
        return top(scores, shortlist_positions)

    def similarity(self, encoded, query: int, position: int) -> float:
        return float(self.inliers(encoded[1], query, position))

    def memory_bytes(self) -> int:
        return self.base.memory_bytes() + sum(d.nbytes for d in self.descriptors) + sum(k.nbytes for k in self.keypoints)


def search(systems, index_ids: list[str], queries: FeatureSet, true_ids: list[str], kind: str, strength: float) -> list[dict]:
    """One row per query, system and mode; `seconds` is the search time after feature extraction."""
    position = {image_id: i for i, image_id in enumerate(index_ids)}
    rows = []
    for system in systems:
        encoded = system.encode(queries)
        keys = system.keys(encoded)
        for i, (query_id, true_id) in enumerate(zip(queries.ids, true_ids)):
            true_position = position[true_id] if true_id else -1
            start = time.perf_counter()
            exhaustive = (*system.rank(encoded, i), len(index_ids), time.perf_counter() - start)
            start = time.perf_counter()
            candidates = system.index.candidates(keys[i])
            lsh = ((*system.rank(encoded, i, candidates), len(candidates), time.perf_counter() - start) if len(candidates)
                   else (-1, -np.inf, 0, time.perf_counter() - start))
            true_is_candidate = bool(true_position >= 0 and np.isin(true_position, candidates))
            true_similarity = system.similarity(encoded, i, true_position) if true_position >= 0 else np.nan
            for mode, (predicted, score, searched, seconds) in (("exhaustive", exhaustive), ("lsh", lsh)):
                rows.append(dict(system=system.name, mode=mode, kind=kind, strength=strength, query_id=query_id, true_id=true_id,
                                 predicted=index_ids[predicted] if predicted >= 0 else "", score=score, candidates=searched,
                                 true_is_candidate=true_is_candidate, true_similarity=true_similarity, seconds=seconds))
    return rows

## 7. Evaluation at a false-match rate

A system is useful only at a low false-match rate, so each is calibrated on the distractors: the threshold is the lowest score that at most 1% of distractor queries exceed (ties are common for pHash's integer distances, so a plain quantile could overshoot).
A target counts as found if the top answer is its source and scores above the threshold.
Thresholds are set on validation and reused unchanged on test; for Meta's development queries they are set on `dev_calibration` and reported on `dev_report`, with the challenge's micro-average precision (Douze et al., 2021).

Per-condition rates carry Wilson intervals (Wilson, 1927).
Rates pooled over conditions carry 95% intervals from a cluster bootstrap that resamples whole source images, because each source is queried under all 52 conditions and one source resembling an indexed image yields false matches under many edits at once.

In [ ]:
RESULT_COLUMNS = ["system", "mode", "kind", "strength", "query_id", "true_id", "predicted", "score", "candidates",
                  "true_is_candidate", "true_similarity", "seconds"]
GROUP = ["system", "mode", "kind", "strength"]
Z_95 = 1.959963984540054


def is_target(frame: pd.DataFrame) -> pd.Series:
    return frame.true_id != ""


def wilson_interval(successes, n, z: float = Z_95):
    successes, n = np.asarray(successes, float), np.asarray(n, float)
    p, denominator = successes / n, 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denominator
    half = z / denominator * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return center - half, center + half


def source_resamples(source, values: np.ndarray, resamples: int = 2000, seed: int = 0) -> np.ndarray:
    """Column totals of `values` in bootstrap samples that draw whole source images with replacement."""
    sources, inverse = np.unique(np.asarray(source), return_inverse=True)
    per_source = np.zeros((len(sources), values.shape[1]))
    np.add.at(per_source, inverse, values)
    counts = np.random.default_rng(seed).multinomial(len(sources), np.full(len(sources), 1 / len(sources)), size=resamples)
    return counts @ per_source


def percentile_interval(samples: np.ndarray) -> tuple[float, float]:
    low, high = np.nanpercentile(samples, [2.5, 97.5])
    return float(low), float(high)


def threshold_allowing(distractor_scores, allowed: int) -> float:
    scores = np.sort(np.asarray(distractor_scores, float))
    return -np.inf if allowed >= len(scores) else float(scores[len(scores) - allowed - 1])


def threshold_at_fmr(distractor_scores, fmr: float) -> float:
    return threshold_allowing(distractor_scores, int(np.floor(fmr * len(distractor_scores) + 1e-9)))


def calibrate(results: pd.DataFrame, fmr: float = TARGET_FMR) -> pd.DataFrame:
    distractors = results[~is_target(results)]
    return (distractors.groupby(["system", "mode"]).score.apply(lambda s: threshold_at_fmr(s.to_numpy(), fmr))
            .rename("threshold").reset_index())


def pooled_rates(results: pd.DataFrame, thresholds: pd.DataFrame, seed: int = 0) -> pd.DataFrame:
    df = results.merge(thresholds, on=["system", "mode"], validate="many_to_one")
    df = df.assign(target=is_target(df), accepted=df.score > df.threshold)
    rows = []
    for (system, mode), group in df.groupby(["system", "mode"], sort=False):
        targets, distractors = group[group.target], group[~group.target]
        hit = (targets.accepted & (targets.predicted == targets.true_id)).to_numpy(float)
        false = distractors.accepted.to_numpy(float)
        recall = source_resamples(targets.query_id, np.column_stack([hit, np.ones_like(hit)]), seed=seed)
        fmr = source_resamples(distractors.query_id, np.column_stack([false, np.ones_like(false)]), seed=seed)
        rows.append(dict(system=system, mode=mode, recall_at_threshold=hit.mean(),
                         **dict(zip(("recall_low", "recall_high"), percentile_interval(recall[:, 0] / recall[:, 1]))),
                         false_match_rate=false.mean(), **dict(zip(("fmr_low", "fmr_high"), percentile_interval(fmr[:, 0] / fmr[:, 1])))))
    return pd.DataFrame(rows)


def micro_average_precision(correct, scores, positives: int) -> tuple[float, float]:
    """As in Meta's evaluation code: all predictions pooled and ranked by score, ties ranked incorrect-first."""
    correct, scores = np.asarray(correct, bool), np.asarray(scores, float)
    order = np.lexsort((correct, -scores))
    true_positives = np.cumsum(correct[order])
    precision = true_positives / np.arange(1, len(order) + 1)
    recall = true_positives / positives
    reachable = precision >= 0.9
    return float((np.diff(recall, prepend=0) * precision).sum()), float(recall[reachable].max()) if reachable.any() else 0.0


def retrieval_scores(results: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (system, mode), group in results.groupby(["system", "mode"]):
        predicted = group[group.predicted != ""]
        ap, r90 = micro_average_precision(predicted.predicted == predicted.true_id, predicted.score, int(is_target(group).sum()))
        rows.append(dict(system=system, mode=mode, micro_ap=ap, recall_at_p90=r90))
    return pd.DataFrame(rows)


def summarize(results: pd.DataFrame, thresholds: pd.DataFrame) -> pd.DataFrame:
    df = results.merge(thresholds, on=["system", "mode"], validate="many_to_one")
    correct, accepted = df.predicted == df.true_id, df.score > df.threshold
    df = df.assign(target=is_target(df), correct=correct, accepted=accepted, accepted_hit=correct & accepted)
    summary = df[df.target].groupby(GROUP, sort=False).agg(queries=("correct", "size"), hits=("correct", "sum"),
                                                           accepted_hits=("accepted_hit", "sum"), candidates=("candidates", "mean"),
                                                           true_similarity=("true_similarity", "mean"))
    false = df[~df.target].groupby(GROUP, sort=False).agg(distractors=("accepted", "size"), false_matches=("accepted", "sum"))
    summary = summary.join(false).reset_index()
    for name, successes, n in (("recall_at_1", summary.hits, summary.queries), ("recall_at_threshold", summary.accepted_hits, summary.queries),
                               ("false_match_rate", summary.false_matches, summary.distractors)):
        summary[name] = successes / n
        summary[f"{name}_low"], summary[f"{name}_high"] = wilson_interval(successes, n)
    return summary


def condition_order(frame: pd.DataFrame) -> pd.DataFrame:
    order = {condition: i for i, condition in enumerate(CONDITIONS)}
    return frame.sort_values("kind", key=lambda _: [order.get(c, -1) for c in zip(frame.kind, frame.strength)], kind="stable")


def by_condition(frame: pd.DataFrame, columns, values) -> pd.DataFrame:
    conditions = pd.MultiIndex.from_frame(frame[["kind", "strength"]].drop_duplicates())
    return frame.pivot_table(index=["kind", "strength"], columns=columns, values=values).reindex(conditions)


STYLES = {(system, mode): (color, "-" if mode == "lsh" else "--")
          for system, color in (("phash", "C0"), ("visual-words", "C1"), ("visual-words+ransac", "C2"))
          for mode in ("lsh", "exhaustive")}


def plot_recall(summary: pd.DataFrame, title: str) -> None:
    kinds = [k for k in dict.fromkeys(summary.kind) if k not in ("clean", "flip")]
    fig, axes = plt.subplots(-(-len(kinds) // 4), 4, figsize=(15, 3.2 * -(-len(kinds) // 4)), squeeze=False)
    for ax, kind in zip(axes.flat, kinds):
        for (system, mode), (color, line) in STYLES.items():
            part = summary[(summary.kind == kind) & (summary.system == system) & (summary["mode"] == mode)].sort_values("strength")
            ax.plot(part.strength.astype(float), part.recall_at_1, color=color, linestyle=line, marker="o", markersize=3, label=f"{system} {mode}")
            ax.fill_between(part.strength.astype(float), part.recall_at_1_low, part.recall_at_1_high, color=color, alpha=0.12, linewidth=0)
        ax.set(title=kind, ylim=(0, 1.02), xlabel="strength", ylabel="recall@1")
    for ax in axes.flat[len(kinds):]:
        ax.axis("off")
    axes.flat[0].legend(fontsize=6)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def plot_s_curves(results: pd.DataFrame) -> None:
    """Observed share of true matches that became candidates, per similarity bin, against the predicted curve."""
    lsh = results[(results["mode"] == "lsh") & is_target(results)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    grid, distances = np.linspace(0, 1, 201), np.arange(65)
    axes[0].plot(1 - distances / 64, bit_sampling_curve(distances, SETTINGS["bit_bands"], SETTINGS["bit_rows"]), "k-", label="predicted")
    axes[1].plot(grid, s_curve(grid, SETTINGS["minhash_bands"], SETTINGS["minhash_rows"]), "k-", label="predicted")
    for ax, system, label in ((axes[0], "phash", "bit agreement 1 - d/64"), (axes[1], "visual-words", "Jaccard similarity")):
        part = lsh[lsh.system == system]
        grouped = part.groupby(pd.cut(part.true_similarity, np.linspace(0, 1, 21)), observed=True).true_is_candidate.agg(["mean", "size"])
        ax.scatter([i.mid for i in grouped.index], grouped["mean"], s=5 + 40 * grouped["size"] / grouped["size"].max(), color="C3",
                   label="measured (size = count)", zorder=3)
        ax.set(title=system, xlabel=label, ylabel="P(true match is a candidate)", ylim=(-0.02, 1.02))
        ax.legend(fontsize=7)
    fig.tight_layout()
    plt.show()

## 8. Running one split

`run` builds the four systems over the index (or a seeded subset of it that always holds the targets, for the scale points), transforms the split's targets and distractors under every condition, searches, calibrates or applies the thresholds, and returns the per-query rows with their summaries.
Every table it shows is also written to `results/notebook/` under the run's tag.

In [ ]:
@dataclass
class Run:
    tag: str
    results: pd.DataFrame
    thresholds: pd.DataFrame
    overall: pd.DataFrame
    summary: pd.DataFrame
    costs: pd.DataFrame
    index_ids: list[str]


def index_subset(index_ids: list[str], required: set[str], size: int | None, seed: int) -> np.ndarray:
    if size is None or size >= len(index_ids):
        return np.arange(len(index_ids))
    must = np.array([i for i, image_id in enumerate(index_ids) if image_id in required], dtype=np.int64)
    others = np.setdiff1d(np.arange(len(index_ids)), must)
    return np.sort(np.concatenate([must, np.random.default_rng(seed).permutation(others)[: size - len(must)]]))


def run_tag(index_size: int, seed: int) -> str:
    s = SETTINGS
    return (f"{DATASET}_n{index_size}_k{s['vocabulary']}_p{s['num_perm']}_mb{s['minhash_bands']}r{s['minhash_rows']}"
            f"_bb{s['bit_bands']}r{s['bit_rows']}_t{s['top_k']}_s{seed}")


def query_sets(split: str) -> list[tuple[str, float, tuple, list[str], list[str]]]:
    """(kind, strength, edit, query IDs, true IDs) per query set; distractors have an empty true ID."""
    if split == "dev":
        truth = dev_ground_truth()
        return [(part, 0, ("clean", 0), ids, [truth.get(q, "") for q in ids])
                for part in ("dev_calibration", "dev_report") for ids in [SPLITS[part][:QUERY_LIMIT]]]
    targets, distractors = SPLITS[f"{split}_targets"][:QUERY_LIMIT], SPLITS[f"{split}_distractors"][:QUERY_LIMIT]
    return [(kind, strength, (kind, strength), targets + distractors, targets + [""] * len(distractors)) for kind, strength in CONDITIONS]


def run(split: str, seed: int = 0, index_size: int | None = None, scale: bool = False, show: bool = True) -> Run:
    full = index_features(scale)
    vocab = vocabulary(SETTINGS["vocabulary"], seed)
    sets = query_sets(split)
    positions = index_subset(full.ids, {t for s in sets for t in s[4] if t}, index_size, seed)
    index = full.subset(positions)
    words = VisualWordSystem(index.descriptors, vocab, SETTINGS["num_perm"], SETTINGS["minhash_bands"], SETTINGS["minhash_rows"], seed)
    systems = [PhashSystem(index.hashes, SETTINGS["bit_bands"], SETTINGS["bit_rows"], seed), words,
               SpatiallyVerifiedSystem(words, index.descriptors, index.keypoints, SETTINGS["top_k"]), ContainmentSystem(words)]
    indexed, rows = set(index.ids), []
    for kind, strength, (edit, edit_strength), query_ids, true_ids in tqdm(sets, desc=f"{split} search"):
        jobs = [(DATASET, q, edit, edit_strength) for q in query_ids]
        queries = cached_features(f"queries/{split}_{kind}_{strength}_n{len(jobs)}", jobs, f"{kind} {strength}")
        rows += search(systems, index.ids, queries, [t if t in indexed else "" for t in true_ids], kind, strength)
    results = pd.DataFrame(rows, columns=RESULT_COLUMNS)

    run_name = run_tag(len(index.ids), seed) + (f"_q{QUERY_LIMIT}" if QUERY_LIMIT else "")
    tag = f"{run_name}_{split}"
    threshold_file = OUT / f"thresholds_{run_name}_fmr{TARGET_FMR}.csv"
    if split == "dev":
        thresholds = calibrate(results[results.kind == "dev_calibration"])
    elif split == "test":
        thresholds = pd.read_csv(threshold_file)
    else:
        thresholds = calibrate(results)
        thresholds.to_csv(threshold_file, index=False)
    reported = results[results.kind == "dev_report"] if split == "dev" else results
    summary = condition_order(summarize(reported, thresholds))
    overall = pooled_rates(reported, thresholds, seed)
    if split == "dev":
        overall = overall.merge(retrieval_scores(reported), on=["system", "mode"])
    per_mode = results.groupby(["system", "mode"]).agg(candidates=("candidates", "mean"), ms_per_query=("seconds", "mean")).reset_index()
    per_mode["ms_per_query"] *= 1000
    costs = pd.DataFrame({"system": [s.name for s in systems], "index_size": len(index.ids),
                          "memory_mb": [s.memory_bytes() / 1e6 for s in systems]}).merge(per_mode, on="system")
    costs["fraction_of_index"] = costs.candidates / costs.index_size
    for name, frame in (("queries", results), ("summary", summary), ("overall", overall), ("costs", costs)):
        frame.to_csv(OUT / f"{name}_{tag}.csv", index=False)
    if show:
        print(f"{tag}: {len(index.ids):,} indexed images, {len(sets)} query sets of {len(sets[0][3]):,} queries")
        print(f"Recall at the {TARGET_FMR:.0%} false-match threshold, 95% intervals over source images:")
        display(overall.round(4))
        print("Search cost per query and index memory:")
        display(costs.round(4))
    return Run(tag, results, thresholds, overall, summary, costs, index.ids)

## 9. Choosing every LSH setting without searching (tuning on validation)

The s-curve predicts which true pairs LSH keeps, so every setting can be scored from one validation run.
For each validation query we compute its similarity to every indexed image once (Hamming distance for pHash, Jaccard similarity for each vocabulary size) and predict, for every (b, r):

- recall at the threshold with LSH, as the mean over targets of [accepted by exhaustive search] x P(candidate | true similarity);
- the candidate fraction, as the mean of P(candidate | similarity) over all query-image pairs;
- for spatial verification, whether the true image enters the top-k shortlist and has more inliers than the threshold.

The run's own settings are predicted too, against what it measured, so the error of the method is printed beside its choices.
We take the cheapest setting within 0.005 of exhaustive recall at the threshold.

In [ ]:
@dataclass
class Profile:
    true_similarity: np.ndarray
    rank: np.ndarray
    top_correct: np.ndarray
    top_score: np.ndarray
    histogram: np.ndarray
    values: np.ndarray

    def candidate_fraction(self, curve) -> float:
        return float((self.histogram * curve(self.values)).sum() / self.histogram.sum())


JACCARD_BINS = 10_000
JACCARD_VALUES = np.concatenate([[0.0], (np.arange(JACCARD_BINS) + 0.5) / JACCARD_BINS])


def summarize_rows(similarity: np.ndarray, true_positions: np.ndarray):
    rows, target = np.arange(len(similarity)), true_positions >= 0
    true_similarity = np.where(target, similarity[rows, np.maximum(true_positions, 0)], np.nan)
    best = similarity.argmax(axis=1)
    return (true_similarity, np.where(target, (similarity > true_similarity[:, None]).sum(axis=1), -1),
            target & (best == true_positions), similarity[rows, best])


def jaccard_profile(index: np.ndarray, sizes: np.ndarray, queries: np.ndarray, true_positions: np.ndarray) -> Profile:
    intersection = queries @ index.T
    similarity = intersection / np.maximum(queries.sum(axis=1)[:, None] + sizes[None, :] - intersection, 1)
    bins = np.where(similarity > 0, 1 + np.minimum(similarity * JACCARD_BINS, JACCARD_BINS - 1).astype(np.int64), 0)
    return Profile(*summarize_rows(similarity, true_positions), np.bincount(bins.ravel(), minlength=JACCARD_BINS + 1), JACCARD_VALUES)


def hamming_profile(index: np.ndarray, queries: np.ndarray, true_positions: np.ndarray) -> Profile:
    distance = np.bitwise_count(index[None, :] ^ queries[:, None]).astype(np.int64)
    return Profile(*summarize_rows(-distance, true_positions), np.bincount(distance.ravel(), minlength=65), np.arange(65))


def concatenate_profiles(parts: list[Profile]) -> Profile:
    return Profile(*(np.concatenate([getattr(p, f) for p in parts]) for f in ("true_similarity", "rank", "top_correct", "top_score")),
                   sum(p.histogram for p in parts), parts[0].values)


def accepted_at(profile: Profile, target: np.ndarray) -> np.ndarray:
    return target & profile.top_correct & (profile.top_score > threshold_at_fmr(profile.top_score[~target], TARGET_FMR))


def bit_sampling_grid(profile: Profile, accepted: np.ndarray, target: np.ndarray) -> pd.DataFrame:
    distance = np.rint(-profile.true_similarity[target]).astype(int)
    rows = [dict(system="phash", bands=b, rows=r, predicted_recall=float((accepted[target] * bit_sampling_curve(distance, b, r)).mean()),
                 candidate_fraction=profile.candidate_fraction(lambda d: bit_sampling_curve(d, b, r)))
            for r in range(4, 17) for b in range(1, 65)]
    return pd.DataFrame(rows).assign(exhaustive_recall=accepted[target].mean())


def minhash_grid(profile: Profile, system: str, accepted: np.ndarray, target: np.ndarray, num_perms=(128, 256)) -> pd.DataFrame:
    jaccard = profile.true_similarity[target]
    rows = [dict(system=system, num_perm=p, bands=b, rows=r, predicted_recall=float((accepted[target] * s_curve(jaccard, b, r)).mean()),
                 candidate_fraction=profile.candidate_fraction(lambda s: s_curve(s, b, r)))
            for p in num_perms for r in range(1, 9) for b in range(1, p // r + 1)]
    return pd.DataFrame(rows).assign(exhaustive_recall=accepted[target].mean())


def cheapest(grid: pd.DataFrame, tolerance: float = 0.005) -> pd.DataFrame:
    keys = [c for c in ("system", "vocabulary", "top_k") if c in grid]
    good = grid[grid.predicted_recall >= grid.exhaustive_recall - tolerance]
    return good.sort_values(["candidate_fraction", "bands"]).groupby(keys, dropna=False).head(1).sort_values(keys)


def tune(val: Run, vocabularies: list[int], top_ks=(5, 10, 25, 50)) -> pd.DataFrame:
    seed = int(val.tag.split("_s")[-1].split("_")[0])
    vocabs = {k: vocabulary(k, seed) for k in vocabularies}
    full_position = {image_id: i for i, image_id in enumerate(INDEX.ids)}
    index = INDEX.subset(np.array([full_position[i] for i in val.index_ids]))
    dense = {k: incidence(word_sets(index.descriptors, v), k).toarray() for k, v in vocabs.items()}
    position = {image_id: i for i, image_id in enumerate(index.ids)}
    targets, distractors = SPLITS["val_targets"][:QUERY_LIMIT], SPLITS["val_distractors"][:QUERY_LIMIT]
    ids = targets + distractors
    true_positions = np.array([position[t] for t in targets] + [-1] * len(distractors))
    hamming_parts, jaccard_parts, keys = [], {k: [] for k in vocabs}, []
    for kind, strength in tqdm(CONDITIONS, desc="similarity profiles"):
        queries = cached_features(f"queries/val_{kind}_{strength}_n{len(ids)}", [(DATASET, q, kind, strength) for q in ids], kind)
        hamming_parts.append(hamming_profile(index.hashes, queries.hashes, true_positions))
        for k, v in vocabs.items():
            jaccard_parts[k].append(jaccard_profile(dense[k], dense[k].sum(axis=1), incidence(word_sets(queries.descriptors, v), k).toarray(),
                                                    true_positions))
        keys += [(kind, float(strength), q) for q in ids]
    target = np.tile(true_positions >= 0, len(CONDITIONS))
    hamming = concatenate_profiles(hamming_parts)
    jaccard = {k: concatenate_profiles(parts) for k, parts in jaccard_parts.items()}

    results = val.results.assign(strength=val.results.strength.astype(float))
    verified = results[(results.system == "visual-words+ransac") & (results["mode"] == "exhaustive")]
    inlier_threshold = threshold_at_fmr(verified.score[~is_target(verified)].to_numpy(), TARGET_FMR)
    inliers = verified.set_index(["kind", "strength", "query_id"]).true_similarity.reindex(keys).to_numpy()
    grids, study = [bit_sampling_grid(hamming, accepted_at(hamming, target), target)], []
    for k, profile in jaccard.items():
        accepted = accepted_at(profile, target)
        grids.append(minhash_grid(profile, "visual-words", accepted, target).assign(vocabulary=k))
        shortlisted = {t: target & (profile.rank >= 0) & (profile.rank < t) & (inliers > inlier_threshold) for t in top_ks}
        grids += [minhash_grid(profile, "visual-words+ransac", shortlisted[t], target).assign(vocabulary=k, top_k=t) for t in top_ks]
        study.append(dict(vocabulary=k, davies_bouldin=vocabs[k].davies_bouldin, mean_true_jaccard=np.nanmean(profile.true_similarity),
                          recall_at_1=profile.top_correct[target].mean(), recall_at_threshold=accepted[target].mean(),
                          **{f"verified_top{t}": shortlisted[t][target].mean() for t in top_ks}))
    grid = pd.concat(grids, ignore_index=True)
    grid.to_csv(OUT / f"tuning_{val.tag}.csv", index=False)

    measured = lambda system: results[(results.system == system) & (results["mode"] == "lsh")].candidates.mean() / len(index.ids)
    own = jaccard[SETTINGS["vocabulary"]]
    measured_verified = ((verified.predicted == verified.true_id) & (verified.score > inlier_threshold))[is_target(verified)].mean()
    shortlisted = target & (own.rank >= 0) & (own.rank < SETTINGS["top_k"]) & (inliers > inlier_threshold)
    print("The method checked against the run's own settings (measured, predicted):")
    display(pd.DataFrame([
        ("phash candidate fraction", measured("phash"),
         hamming.candidate_fraction(lambda d: bit_sampling_curve(d, SETTINGS["bit_bands"], SETTINGS["bit_rows"]))),
        ("visual-words candidate fraction", measured("visual-words"),
         own.candidate_fraction(lambda s: s_curve(s, SETTINGS["minhash_bands"], SETTINGS["minhash_rows"]))),
        ("visual-words+ransac exhaustive recall at the threshold", measured_verified, shortlisted[target].mean()),
    ], columns=["quantity", "measured", "predicted"]).round(4))
    print("Vocabulary study (validation, exhaustive retrieval):")
    display(pd.DataFrame(study).round(3))
    print("Cheapest LSH within 0.005 of exhaustive recall at the threshold:")
    display(cheapest(grid).round(4))
    return grid

## 10. LSH loss and the cascade (no new search)

**LSH loss.** LSH finds a target that exhaustive search finds exactly when the true match becomes a candidate, so the predicted LSH recall is the mean over targets of [exhaustive correct] x P(candidate | true similarity).

**The cascade.** pHash answers if its best distance is below a threshold t_A; otherwise System B answers if its score exceeds t_B.
The two thresholds are calibrated jointly: for each t_A, the distractors pHash accepts spend part of the false-match budget, and t_B is the loosest that keeps the rest within it.
It is compared with each system alone, with score fusion (each score turned into its tail probability among calibration distractors), and with the best possible combination, which counts a target as found if either system finds it.
A test run's combinations are calibrated on its validation run and evaluated on the whole test split, the protocol of the single systems; the development queries are calibrated on `dev_calibration` and evaluated on `dev_report`, and a validation run on half of its source images and the other half.

In [ ]:
def candidate_probability(results: pd.DataFrame) -> np.ndarray:
    pairs = results.true_similarity.to_numpy(float)
    probability = np.full(len(results), np.nan)
    phash = (results.system == "phash").to_numpy() & ~np.isnan(pairs)
    words = (results.system == "visual-words").to_numpy() & ~np.isnan(pairs)
    probability[phash] = bit_sampling_curve(np.rint(64 * (1 - pairs[phash])).astype(int), SETTINGS["bit_bands"], SETTINGS["bit_rows"])
    probability[words] = s_curve(pairs[words], SETTINGS["minhash_bands"], SETTINGS["minhash_rows"])
    return probability


def lsh_loss(results: pd.DataFrame) -> pd.DataFrame:
    targets = results[is_target(results) & results.system.isin(["phash", "visual-words"])]
    keys = ["system", "kind", "strength", "query_id"]
    exhaustive = targets[targets["mode"] == "exhaustive"].set_index(keys)
    lsh = targets[targets["mode"] == "lsh"].set_index(keys)
    lsh = lsh.assign(exhaustive_correct=(exhaustive.predicted == exhaustive.true_id).reindex(lsh.index), lsh_correct=lsh.predicted == lsh.true_id,
                     probability=candidate_probability(lsh.reset_index()))
    lsh["predicted_lsh_correct"] = lsh.exhaustive_correct * lsh.probability
    return condition_order(lsh.groupby(["system", "kind", "strength"], sort=False).agg(
        queries=("lsh_correct", "size"), exhaustive_recall=("exhaustive_correct", "mean"), lsh_recall=("lsh_correct", "mean"),
        predicted_lsh_recall=("predicted_lsh_correct", "mean"), candidate_rate=("true_is_candidate", "mean"),
        predicted_candidate_rate=("probability", "mean")).reset_index())


PAIR_KEYS = ["mode", "kind", "strength", "query_id", "true_id"]


def paired(results: pd.DataFrame, mode: str, system_b: str) -> pd.DataFrame:
    part = results[results["mode"] == mode]
    a = part[part.system == "phash"].set_index(PAIR_KEYS)[["predicted", "score"]]
    b = part[part.system == system_b].set_index(PAIR_KEYS)[["predicted", "score"]]
    return a.add_suffix("_a").join(b.add_suffix("_b"), validate="one_to_one").reset_index()


def cascade_answer(pairs: pd.DataFrame, t_a: float, t_b: float):
    settled_by_a = pairs.score_a > t_a
    accepted_b = ~settled_by_a & (pairs.score_b > t_b)
    return pairs.predicted_a.where(settled_by_a, pairs.predicted_b.where(accepted_b, "")), settled_by_a


def calibrate_cascade(pairs: pd.DataFrame, fmr: float = TARGET_FMR) -> tuple[float, float]:
    """The threshold pair with the highest target recall whose distractor false matches stay within `fmr`;
    ties go to the pair that settles more queries with pHash alone, which saves SIFT extraction."""
    distractors, targets = pairs[~is_target(pairs)], pairs[is_target(pairs)]
    budget = int(np.floor(fmr * len(distractors) + 1e-9))
    best, best_key = None, None
    for t_a in np.unique(np.concatenate([pairs.score_a.to_numpy(float), [np.inf]])):
        passed_on = distractors.score_a <= t_a
        spent = int((~passed_on).sum())
        if spent > budget:
            continue
        t = (float(t_a), threshold_allowing(distractors.score_b[passed_on].to_numpy(), budget - spent))
        answer, settled = cascade_answer(targets, *t)
        key = ((answer == targets.true_id).sum(), settled.sum())
        if best_key is None or key > best_key:
            best, best_key = t, key
    return best


def tail(sorted_scores: np.ndarray, scores: pd.Series) -> np.ndarray:
    return 1 - np.searchsorted(sorted_scores, scores.to_numpy(float), side="left") / len(sorted_scores)


def fused(pairs: pd.DataFrame, sorted_a: np.ndarray, sorted_b: np.ndarray):
    tail_a, tail_b = tail(sorted_a, pairs.score_a), tail(sorted_b, pairs.score_b)
    return pairs.predicted_a.where(tail_a <= tail_b, pairs.predicted_b), -np.minimum(tail_a, tail_b)


def compare(pairs: pd.DataFrame, system_b: str, calibrating: pd.Series | None = None, fmr: float = TARGET_FMR):
    if calibrating is None:  # by source image, so a source and all its edited versions fall in one half
        calibrating = pairs.query_id.map(lambda q: zlib.crc32(q.encode()) % 2 == 0)
    fit, held = pairs[calibrating], pairs[~calibrating]
    fit_distractors = fit[~is_target(fit)]
    t_a, t_b = threshold_at_fmr(fit_distractors.score_a.to_numpy(), fmr), threshold_at_fmr(fit_distractors.score_b.to_numpy(), fmr)
    cascade = calibrate_cascade(fit, fmr)
    sorted_a, sorted_b = np.sort(fit_distractors.score_a.to_numpy(float)), np.sort(fit_distractors.score_b.to_numpy(float))
    fusion_threshold = threshold_at_fmr(fused(fit_distractors, sorted_a, sorted_b)[1], fmr)

    answer_a = held.predicted_a.where(held.score_a > t_a, "")
    answer_b = held.predicted_b.where(held.score_b > t_b, "")
    answer_cascade, settled_by_a = cascade_answer(held, *cascade)
    fusion_answer, fusion_score = fused(held, sorted_a, sorted_b)
    answers = {"phash": answer_a, system_b: answer_b, "cascade": answer_cascade, "fusion": fusion_answer.where(fusion_score > fusion_threshold, ""),
               "either": answer_a.where((answer_a == held.true_id) & is_target(held) | (answer_b == ""), answer_b)}
    settled = {"phash": 1.0, system_b: 0.0, "cascade": settled_by_a.mean(), "fusion": 0.0, "either": 0.0}
    target = is_target(held)
    per_condition, found, false = [], {}, {}
    for method, answer in answers.items():
        found[method] = ((answer == held.true_id) & target)[target].to_numpy(float)
        false[method] = (answer[~target] != "").to_numpy(float)
        counts = held[target][["kind", "strength"]].assign(found=found[method] > 0)
        part = counts.groupby(["kind", "strength"], sort=False).found.agg(hits="sum", queries="size").reset_index()
        part["recall"] = part.hits / part.queries
        part["recall_low"], part["recall_high"] = wilson_interval(part.hits, part.queries)
        per_condition.append(part.assign(method=method))
    methods = list(answers)
    recall = source_resamples(held.query_id[target], np.column_stack([found[m] for m in methods])) / target.sum()
    fmr_samples = source_resamples(held.query_id[~target], np.column_stack([false[m] for m in methods])) / (~target).sum()
    summary = pd.DataFrame([dict(method=m, mean_recall=found[m].mean(), **dict(zip(("recall_low", "recall_high"), percentile_interval(recall[:, i]))),
                                 false_match_rate=false[m].mean(), **dict(zip(("fmr_low", "fmr_high"), percentile_interval(fmr_samples[:, i]))),
                                 settled_without_b=settled[m]) for i, m in enumerate(methods)])
    return condition_order(pd.concat(per_condition, ignore_index=True)), summary


def compare_calibrated_on(calibration: pd.DataFrame, evaluation: pd.DataFrame, system_b: str, fmr: float = TARGET_FMR):
    """`compare` with every threshold set on the `calibration` pairs and every rate measured on all `evaluation` pairs."""
    both = pd.concat([calibration, evaluation], ignore_index=True)
    return compare(both, system_b, pd.Series(np.arange(len(both)) < len(calibration), index=both.index), fmr)


def operating_points(calibration: Run, evaluation: Run, system_b: str, fmrs=(0.001, 0.002, 0.005, 0.01, 0.02, 0.05)) -> None:
    """Recall against the false-match rate each method realises on `evaluation` when calibrated on `calibration` at
    several target rates: one target realises different rates for different methods, so the curves compare them
    at equal realised rates. The points calibrated at the 1% target are circled."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, mode in zip(axes, ("exhaustive", "lsh")):
        fit, held = paired(calibration.results, mode, system_b), paired(evaluation.results, mode, system_b)
        points = pd.concat([compare_calibrated_on(fit, held, system_b, fmr)[1].assign(target_fmr=fmr) for fmr in fmrs])
        points.to_csv(OUT / f"operating_points_{evaluation.tag}_{system_b}_{mode}.csv", index=False)
        for method, curve in points.groupby("method", sort=False):
            ax.plot(curve.false_match_rate, curve.mean_recall, marker=".", label=method)
            at_target = curve[np.isclose(curve.target_fmr, TARGET_FMR)]
            ax.scatter(at_target.false_match_rate, at_target.mean_recall, s=60, facecolors="none", edgecolors="k")
        ax.axvline(TARGET_FMR, color="0.6", linewidth=0.8, linestyle=":")
        ax.set(xscale="log", title=f"{mode} retrieval", xlabel="realised false-match rate", ylabel="recall at threshold")
        ax.legend(fontsize=7)
    fig.tight_layout()
    plt.show()


def analyze(r: Run, calibration: Run | None = None, show_conditions: bool = False) -> dict:
    """LSH loss and every combination for one run, calibrated on `calibration` if given (the validation run of a
    test run); returns the combination summaries by (system B, mode)."""
    loss = lsh_loss(r.results)
    loss.to_csv(OUT / f"lsh_loss_{r.tag}.csv", index=False)
    print("LSH loss, means over conditions (recall@1):")
    display(loss.groupby("system")[["exhaustive_recall", "lsh_recall", "predicted_lsh_recall", "candidate_rate",
                                    "predicted_candidate_rate"]].mean().round(3))
    combinations = {}
    for system_b in ("visual-words", "visual-words+ransac", "visual-words-containment"):
        for mode in ("lsh", "exhaustive"):
            pairs = paired(r.results, mode, system_b)
            if calibration is not None:
                per_condition, summary = compare_calibrated_on(paired(calibration.results, mode, system_b), pairs, system_b)
            else:
                per_condition, summary = compare(pairs, system_b, pairs.kind == "dev_calibration" if r.tag.endswith("_dev") else None)
            per_condition.to_csv(OUT / f"combinations_{r.tag}_{system_b}_{mode}.csv", index=False)
            summary.to_csv(OUT / f"combinations_summary_{r.tag}_{system_b}_{mode}.csv", index=False)
            combinations[system_b, mode] = summary
            if show_conditions:
                display(by_condition(per_condition, "method", "recall")[summary.method].round(2))
    table = pd.concat({f"{b}, {m}": s.set_index("method").mean_recall for (b, m), s in combinations.items()}, axis=1).T
    print("Recall at the common false-match rate on the evaluated queries, per B system and retrieval mode:")
    display(table.round(3))
    return combinations

## 11. Why each system fails

Per condition, the median over targets of what each representation keeps of the true pair: the Hamming distance for pHash, the keypoints left in the query and the Jaccard similarity for visual words, and the ratio-test matches and RANSAC inliers for spatial verification (matches on a sample of 100 targets per condition, since they are recomputed here).
The gallery shows sampled misses at the first strength where each system's recall@1 drops below 0.5, with the true source and the wrong answer; it is written under `data/failures/` with the images' attributions and never published with the notebook.

In [ ]:
def mechanisms(r: Run, split: str, sample_size: int = 100, seed: int = 0) -> pd.DataFrame:
    position = {image_id: i for i, image_id in enumerate(INDEX.ids)}
    targets, distractors = SPLITS[f"{split}_targets"][:QUERY_LIMIT], SPLITS[f"{split}_distractors"][:QUERY_LIMIT]
    sample = np.random.default_rng(seed).choice(len(targets), size=min(sample_size, len(targets)), replace=False)
    exhaustive = r.results[(r.results["mode"] == "exhaustive") & is_target(r.results)]
    exhaustive = exhaustive.assign(strength=exhaustive.strength.astype(float), correct=exhaustive.predicted == exhaustive.true_id)
    true = exhaustive.pivot_table(index=["kind", "strength"], columns="system", values="true_similarity", aggfunc="median")
    recall = exhaustive.pivot_table(index=["kind", "strength"], columns="system", values="correct", aggfunc="mean")
    rows = []
    with ThreadPoolExecutor(WORKERS) as pool:
        for kind, strength in CONDITIONS:
            ids = targets + distractors
            queries = cached_features(f"queries/{split}_{kind}_{strength}_n{len(ids)}", [(DATASET, q, kind, strength) for q in ids], kind)
            source = lambda i: position[targets[i]]
            matches = list(pool.map(lambda i: verify(queries.keypoints[i], INDEX.keypoints[source(i)], queries.descriptors[i],
                                                     INDEX.descriptors[source(i)])[0], sample.tolist()))
            key = (kind, float(strength))
            rows.append(dict(kind=kind, strength=strength, phash_distance=64 * (1 - true.loc[key, "phash"]), phash_recall=recall.loc[key, "phash"],
                             query_keypoints=float(np.median([len(d) for d in queries.descriptors[: len(targets)]])),
                             true_jaccard=true.loc[key, "visual-words"], words_recall=recall.loc[key, "visual-words"],
                             ratio_matches=float(np.median(matches)), inliers=true.loc[key, "visual-words+ransac"],
                             ransac_recall=recall.loc[key, "visual-words+ransac"]))
    table = pd.DataFrame(rows)
    table.to_csv(OUT / f"mechanisms_{r.tag}.csv", index=False)
    return table


def gallery(r: Run, system: str, per_condition: int = 2, seed: int = 0) -> set[str]:
    """Save sampled misses of `system` where its recall@1 first drops below 0.5; returns the images shown."""
    part = r.results[(r.results.system == system) & (r.results["mode"] == "exhaustive") & is_target(r.results)]
    recall = (part.predicted == part.true_id).groupby([part.kind, part.strength.astype(float)]).mean()
    breaking = {}
    for kind, strength in CONDITIONS:
        if kind not in breaking and recall.get((kind, float(strength)), 1) < 0.5:
            breaking[kind] = (kind, strength)
    rng, chosen = np.random.default_rng(seed), []
    for kind, strength in breaking.values():
        misses = part[(part.kind == kind) & (part.strength.astype(float) == float(strength)) & (part.predicted != part.true_id)]
        chosen += [(strength, misses.iloc[i]) for i in rng.choice(len(misses), size=min(per_condition, len(misses)), replace=False)]
    if not chosen:
        return set()
    fig, axes = plt.subplots(len(chosen), 3, figsize=(7.5, 2.6 * len(chosen)), squeeze=False)
    for row, (strength, miss) in zip(axes, chosen):
        query = transform(decode(image_path(DATASET, miss.query_id)), miss.kind, strength,
                          np.random.default_rng(query_seed(miss.kind, strength, miss.query_id)))
        panels = ((query, f"{miss.kind} {strength}"), (decode(image_path(DATASET, miss.predicted)), f"answer {miss.predicted}"),
                  (decode(image_path(DATASET, miss.true_id)), f"source {miss.true_id}"))
        for ax, (img, title) in zip(row, panels):
            ax.imshow(img)
            ax.set_title(title, fontsize=7)
            ax.axis("off")
    fig.tight_layout()
    out = DATA_DIR / "failures" / r.tag
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f"{system}.png", dpi=110)
    plt.close(fig)
    return {m.query_id for _, m in chosen} | {m.predicted for _, m in chosen}

## 12. Deduplication by graph clustering

1,000 random indexed images (300 on Caltech-101) each get three copies with distinct realistic edits, planted into a catalogue of 10,000 DISC21 images.
Each LSH index joins the catalogue with itself, and pairs above a similarity threshold become edges; visual-word pairs with Jaccard similarity of at least 0.1 are also verified with RANSAC, giving edges weighted by inliers.
Each graph is clustered by connected components, which chain two groups through a single false edge, and by Louvain (Blondel et al., 2008), which maximises modularity (Newman and Girvan, 2004) at resolutions 1, 10 and 100, since modularity's resolution limit (Fortunato and Barthelemy, 2007) merges groups as small as these at resolution 1.
Partitions are scored against the planted groups by the adjusted Rand index (Hubert and Arabie, 1985).
Thresholds are chosen on seed 0 and reported on seed 1, a new catalogue with new planted groups.

In [ ]:
COPY_CONDITIONS = (("jpeg", 30), ("downscale", 0.5), ("crop", 0.75), ("rotate", 5), ("overlay", 0.15), ("brightness", 0.5),
                   ("crop+jpeg+overlay", 1), ("flip", 1))


def self_join(system, encoded, similarity_of, floor: float) -> pd.DataFrame:
    keys, parts = system.keys(encoded), []
    for i in range(len(keys)):
        candidates = system.index.candidates(keys[i])
        candidates = candidates[candidates > i]
        similarity = similarity_of(system.scores(encoded, i, candidates))
        kept = similarity >= floor
        parts.append(pd.DataFrame({"i": i, "j": candidates[kept], "similarity": similarity[kept]}))
    return pd.concat(parts, ignore_index=True)


def pairs_together(*labelings: np.ndarray) -> int:
    _, counts = np.unique(np.stack(labelings), axis=1, return_counts=True)
    return int((counts * (counts - 1) // 2).sum())


def evaluate_graph(kept: pd.DataFrame, labels: np.ndarray, seed: int, resolutions=(1, 10, 100)) -> list[dict]:
    n = len(labels)
    partitions = {"connected components": connected_components(sp.coo_matrix((np.ones(len(kept)), (kept.i, kept.j)), shape=(n, n)), directed=False)[1]}
    graph = nx.Graph()
    graph.add_nodes_from(range(n))
    graph.add_weighted_edges_from(kept[["i", "j", "similarity"]].itertuples(index=False, name=None))
    for resolution in resolutions:
        louvain = np.empty(n, np.int64)
        for c, members in enumerate(nx.community.louvain_communities(graph, weight="weight", resolution=resolution, seed=seed)):
            louvain[list(members)] = c
        partitions[f"louvain r={resolution:g}"] = louvain
    rows = []
    for method, clusters in partitions.items():
        both, predicted, planted = pairs_together(labels, clusters), pairs_together(clusters), pairs_together(labels)
        order = np.argsort(clusters, kind="stable")
        communities = [set(c.tolist()) for c in np.split(order, np.flatnonzero(np.diff(clusters[order])) + 1)]
        rows.append(dict(method=method, edges=len(kept), clusters=len(np.unique(clusters)), largest=int(np.bincount(clusters).max()),
                         ari=adjusted_rand_score(labels, clusters), pair_precision=both / predicted if predicted else 1.0,
                         pair_recall=both / planted if planted else 1.0,
                         modularity=nx.community.modularity(graph, communities) if len(kept) else 0.0))
    return rows


def dedup(seed: int, index_size: int | None, groups: int, copies: int) -> pd.DataFrame:
    positions = index_subset(INDEX.ids, set(), index_size, seed)
    index_ids = [INDEX.ids[i] for i in positions]
    rng = np.random.default_rng(seed)
    sources = np.sort(rng.choice(len(index_ids), size=groups, replace=False))
    jobs, labels = [], list(range(len(index_ids)))
    for source in sources:
        for c in rng.choice(len(COPY_CONDITIONS), size=copies, replace=False):
            jobs.append((DATASET, index_ids[source], *COPY_CONDITIONS[c]))
            labels.append(int(source))
    labels = np.array(labels)
    copy_features = cached_features(f"dedup/copies_n{len(index_ids)}_g{groups}_c{copies}_s{seed}", jobs, "copies")
    catalogue = INDEX.subset(positions).concatenate(copy_features)
    phash = PhashSystem(catalogue.hashes, SETTINGS["bit_bands"], SETTINGS["bit_rows"], seed)
    words = VisualWordSystem(catalogue.descriptors, vocabulary(SETTINGS["vocabulary"], seed), SETTINGS["num_perm"],
                             SETTINGS["minhash_bands"], SETTINGS["minhash_rows"], seed)
    hamming, jaccard = [2, 4, 6, 8, 10, 12, 14, 16], [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1]
    union = [(8, 0.3), (10, 0.3), (12, 0.25), (14, 0.2)]
    phash_edges = self_join(phash, catalogue.hashes, lambda s: 1 + s / 64, 1 - max(hamming) / 64)
    word_edges = self_join(words, words.encode(catalogue), lambda s: s, min(jaccard))
    graphs = [("phash", f"d<={h}", phash_edges[phash_edges.similarity >= 1 - h / 64]) for h in hamming]
    graphs += [("visual-words", f"J>={j:g}", word_edges[word_edges.similarity >= j]) for j in jaccard]
    graphs += [("union", f"d<={h} or J>={j:g}", pd.concat([phash_edges[phash_edges.similarity >= 1 - h / 64], word_edges[word_edges.similarity >= j]])
                .drop_duplicates(["i", "j"]).assign(similarity=1.0)) for h, j in union]
    to_verify = word_edges[word_edges.similarity >= 0.1]
    with ThreadPoolExecutor(WORKERS) as pool:  # OpenCV releases the GIL in matching and RANSAC
        inliers = np.fromiter(pool.map(lambda p: verify(catalogue.keypoints[p[0]], catalogue.keypoints[p[1]], catalogue.descriptors[p[0]],
                                                        catalogue.descriptors[p[1]])[1], zip(to_verify.i.tolist(), to_verify.j.tolist())),
                              dtype=np.int64, count=len(to_verify))
    verified = to_verify.assign(similarity=inliers.astype(float))
    for t in (15, 20, 25, 30, 40):
        graphs.append(("verified", f"inliers>={t}", verified[verified.similarity >= t]))
        graphs += [("union verified", f"d<={h} or inliers>={t}", pd.concat([phash_edges[phash_edges.similarity >= 1 - h / 64], verified[verified.similarity >= t]])
                    .drop_duplicates(["i", "j"]).assign(similarity=1.0)) for h in (8, 10)]
    table = pd.DataFrame([dict(system=system, threshold=threshold, **row) for system, threshold, kept in graphs
                          for row in evaluate_graph(kept, labels, seed)])
    table.to_csv(OUT / f"dedup_{DATASET}_n{len(index_ids)}_g{groups}_c{copies}_s{seed}.csv", index=False)
    return table

---
# The experiments

Everything above defines the method; the cells below run it in the order of the report.
Validation numbers only choose settings; the report quotes test, `dev_report` and seed-1 deduplication numbers.

### Validation: calibrate the thresholds, then check the frozen settings by tuning

In [ ]:
val = run("val")
plot_recall(val.summary, f"recall@1 per edit, validation ({DATASET})")
grid = tune(val, EXPERIMENTS["tune_vocabularies"])

In [ ]:
val_combinations = analyze(val)

### Test: the reported retrieval numbers (RQ1, RQ2), with thresholds from validation

In [ ]:
test = run("test")
plot_recall(test.summary, f"recall@1 per edit, test ({DATASET})")
plot_s_curves(test.results)
display(by_condition(test.summary, ["system", "mode"], "recall_at_1").round(2))

### The cascade on test (RQ4)

In [ ]:
test_combinations = analyze(test, calibration=val)
operating_points(val, test, "visual-words+ransac")

### Why each system fails (test)

In [ ]:
display(mechanisms(test, "test").round(3))
shown = set().union(*(gallery(test, system) for system in ("phash", "visual-words", "visual-words+ransac")))
if DATASET == "disc21" and shown:
    attributions = pd.read_csv(DISC21_DIR / "disc21_yfcc_attributions.csv")
    attributions[attributions.disc21_id.isin(shown)].to_csv(DATA_DIR / "failures" / test.tag / "attributions.csv", index=False)
print(f"gallery of {len(shown)} images (local only): {DATA_DIR / 'failures' / test.tag}")

### Meta's development queries: comparison with the published baselines

In [ ]:
if EXPERIMENTS["dev"]:
    dev = run("dev")
    dev_combinations = analyze(dev)

### Scalability (RQ3): cost and recall from 10,000 to 100,000 indexed images (validation queries)

In [ ]:
points = [run("val", index_size=n, show=False) for n in EXPERIMENTS["scale_points"]] + [val]
if EXPERIMENTS["scale_split"]:
    points.append(run("val", scale=True, show=False))
scaling = pd.concat([p.costs.merge(p.overall, on=["system", "mode"]) for p in points])
display(scaling[["system", "mode", "index_size", "memory_mb", "candidates", "ms_per_query", "recall_at_threshold", "false_match_rate"]]
        .sort_values(["system", "mode", "index_size"]).round(4))

### Deduplication (RQ5): choose thresholds on seed 0, report on seed 1

In [ ]:
dedup_settings = EXPERIMENTS["dedup"]
chosen = dedup(0, **dedup_settings)
best = chosen.loc[chosen.groupby(["system", "method"]).ari.idxmax()]
display(best.round(3))
reported = dedup(1, **dedup_settings).merge(best[["system", "threshold", "method"]], on=["system", "threshold", "method"])
display(reported.round(3))

### A second seed: vocabulary, MinHash permutations, bit samples and index subsets drawn anew

In [ ]:
val_s1 = run("val", seed=1, show=False)
test_s1 = run("test", seed=1)
display(pd.concat({"seed 0": test.overall.set_index(["system", "mode"]).recall_at_threshold,
                   "seed 1": test_s1.overall.set_index(["system", "mode"]).recall_at_threshold}, axis=1).round(4))

## Contributions

To be completed by the group: each member's part of the code, the experiments and the report.